[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/15-projects/p07-doc-qa-api/colab_quickstart.ipynb)

# P7 - Policy document Q&A API: quick start
Ask questions about 10 fictional policy PDFs. Hybrid search (BM25 + vectors, fused with RRF) finds the right page; answers cite document and page. CPU runtime is fine; no API key needed.

## 1. Get the code
In Colab this clones the course repo. If you are already inside the project folder (running locally), the clone is skipped.

In [1]:
import os, pathlib, subprocess, sys
PROJECT = "p07-doc-qa-api"
if pathlib.Path("src").is_dir() and pathlib.Path("README.md").exists():
    print("Project folder found - using the local copy, no clone needed.")
else:
    if not pathlib.Path("AI-Engineer").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vvvvvivekkk/AI-Engineer.git"], check=True)
    os.chdir(f"AI-Engineer/15-projects/{PROJECT}")
print("working in", os.getcwd())

Project folder found - using the local copy, no clone needed.
working in /home/claude/AI-Engineer/15-projects/p07-doc-qa-api


## 2. Install the pinned packages (Colab only)
Colab already has PyTorch with GPU support, so we skip the `torch` line to avoid replacing it. Locally you install with `pip install -r requirements.txt` in a venv.

In [2]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    reqs = [l for l in open("requirements.txt") if l.strip() and not l.startswith(("torch==", "#"))]
    open("/tmp/req.txt", "w").writelines(reqs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req.txt"], check=True)
print("ready")

ready


## 3. Generate the PDFs and the golden question set

In [3]:
!python -m src.make_pdfs

wrote 10 PDFs to data/pdfs and 43 golden questions to data/golden.json


## 4. Evaluate retrieval and answers
Compares BM25-only, vector-only and hybrid search on the golden set (Recall@k, MRR), then checks refusals, citations and a faithfulness proxy.

In [4]:
!python -m src.evaluate

Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 37985.87it/s]


mode        R@1    R@3  MRR@3
bm25      0.947  0.974  0.961
vector    0.974  1.000  0.987
hybrid    1.000  1.000  1.000
answered_when_answerable     1.000
refused_when_unanswerable    0.800
citation_accuracy            1.000
faithfulness_proxy           1.000
saved -> data/eval_report.json


## 5. Start the API and ask a real HTTP question
We start `uvicorn` in the background, wait for `/health`, then POST to `/ask`.

In [5]:
import socket, time, httpx
sock = socket.socket(); sock.bind(("127.0.0.1", 0)); PORT = sock.getsockname()[1]; sock.close()   # a free port
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "src.api:app", "--port", str(PORT)],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(120):                      # wait up to ~2 minutes for the index to build
    try:
        print(httpx.get(f"http://127.0.0.1:{PORT}/health").json()); break
    except httpx.HTTPError:
        time.sleep(1)

{'status': 'ok', 'chunks': 22, 'docs_dir': '/home/claude/AI-Engineer/15-projects/p07-doc-qa-api/data/pdfs'}


Ask an answerable question, then one that is not in any document.

In [6]:
for q in ["How many days of paid annual leave do full-time employees get?",
          "What is the Wi-Fi password for guests?",
          "How quickly must a suspected data breach be reported?"]:
    r = httpx.post(f"http://127.0.0.1:{PORT}/ask", json={"question": q}, timeout=60).json()
    print(q, "->", r["answer"], "| confidence", r["confidence"])

How many days of paid annual leave do full-time employees get? -> Full-time employees receive 24 days of paid annual leave per calendar year. [leave-policy.pdf p.1] Up to 5 unused annual leave days may be carried over into the next year. [leave-policy.pdf p.1] | confidence 0.78
What is the Wi-Fi password for guests? -> Passwords must be at least 14 characters long and must never be shared, including with colleagues or IT staff. [security-policy.pdf p.1] | confidence 0.355
How quickly must a suspected data breach be reported? -> A suspected data breach must be reported to the privacy team within 1 hour of discovery. [data-privacy-policy.pdf p.2] | confidence 0.72


Stop the server.

In [7]:
server.terminate(); server.wait(); print('stopped')

stopped


## 6. Run the tests
The tests use FastAPI's `TestClient`, which calls the app in-process.

In [8]:
!python -m pytest -q tests

.

.

...

..                                                                  [100%]


7 passed in 8.44s


## 7. Docker (on your own machine)
Colab cannot run Docker. On a machine with Docker installed:

```bash
docker compose up --build
curl -X POST localhost:8000/ask -H 'Content-Type: application/json' -d '{"question": "How long is the probation period?"}'
```